In [1]:
import json, os, sys
from pathlib import Path

import anthropic
from dotenv import load_dotenv
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

load_dotenv()
SERVER_SCRIPT = Path.cwd() / "mcp_server_week05_Aguilar_Mayorga.py"
MODEL = "claude-sonnet-5-5"

In [2]:
import time

MAX_TURNS = 5                         # limits calls to LLM per question to avoid an infinite loop if the LLM has trouble answering
PRICE_IN, PRICE_OUT = 2.00, 10.00     # USD per million tokens (Sonnet 5.5)

async def run_agent(query: str, effort: str = "low") -> str | None:
    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"],
                                 timeout=60, max_retries=1)   # avoids waiting longer than 1 minute per call
    params = StdioServerParameters(command=sys.executable, args=[str(SERVER_SCRIPT)])
    total_in = total_out = 0

    print("Starting MCP server...", flush = True)
    async with stdio_client(params, errlog = open("server_stderr.log","w")) as (read, write): #changed this line to keep track of whether the connection has been well-stablished
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = (await session.list_tools()).tools
            schemas = [{"name": t.name, "description": t.description or "",
                        "input_schema": t.input_schema} for t in tools]
            print(f"Tools: {[t.name for t in tools]}\nQuery: {query}\n{'='*60}")

            messages = [{"role": "user", "content": query}]
            for turn in range(1, MAX_TURNS + 1):
                t0 = time.time()
                response = client.messages.create(
                    model=MODEL, max_tokens=2048, tools=schemas, messages=messages,
                    thinking={"type": "adaptive", "display": "summarized"},  # see reasoning
                    output_config={"effort": effort},                       # less thinking = lower cost
                )
                u = response.usage
                total_in, total_out = total_in + u.input_tokens, total_out + u.output_tokens
                cost = (total_in * PRICE_IN + total_out * PRICE_OUT) / 1e6
                print(f"\n--- Turn {turn} | {time.time()-t0:.1f}s | in={u.input_tokens} "
                      f"out={u.output_tokens} | accumulated ≈ ${cost:.4f} ---")

                for b in response.content:                 # what it thought and its conclusion
                    if b.type == "thinking" and b.thinking:
                        print(f"[Thinking] {b.thinking}")
                    elif b.type == "text" and b.text:
                        print(f"[Text] {b.text}")
                messages.append({"role": "assistant", "content": response.content})

                if response.stop_reason != "tool_use":
                    final = "".join(b.text for b in response.content if b.type == "text")
                    print(f"\n[stop_reason: {response.stop_reason}]")
                    return final

                tool_results = []
                for block in response.content:
                    if block.type != "tool_use":
                        continue
                    print(f"\n[Tool call] {block.name}\n[Input] {json.dumps(block.input)}")
                    result = await session.call_tool(block.name, block.input)
                    text = "\n".join(c.text for c in result.content if hasattr(c, "text"))
                    print(f"[is_error] {result.is_error}\n[Result]\n{text}")
                    tool_results.append({"type": "tool_result", "tool_use_id": block.id,
                                         "content": text, "is_error": result.is_error})
                messages.append({"role": "user", "content": tool_results})

            print(f"\n[STOPPED] {MAX_TURNS} turns reached without a final answer.")
            return None

# First run: Query on Betelgeuse

In [3]:
QUERY = ("What is the spectral type and parallax of Betelgeuse, and what are the 3 most recent "
         "arXiv papers whose abstracts mention Betelgeuse? Summarize what the papers suggest "
         "about its current evolutionary status.")
answer_betelgeuse = await run_agent(QUERY)

Starting MCP server...
Tools: ['query_simbad', 'search_arxiv']
Query: What is the spectral type and parallax of Betelgeuse, and what are the 3 most recent arXiv papers whose abstracts mention Betelgeuse? Summarize what the papers suggest about its current evolutionary status.

--- Turn 1 | 1.3s | in=861 out=135 | accumulated ≈ $0.0031 ---

[Tool call] query_simbad
[Input] {"object_name": "Betelgeuse"}
[is_error] False
[Result]
{
  "object_type": "s*r",
  "spectral_type": "M1-M2Ia-Iab",
  "parallax_mas": 6.55,
  "radial_velocity_km_s": 21.91
}

[Tool call] search_arxiv
[Input] {"keyword": "Betelgeuse", "max_results": 3}
[is_error] False
[Result]
{
  "arxiv_id": "2610.34540v1",
  "title": "Axions from Betelgeuse: Probing Axion Couplings with $^{23}\\rm{Na}$ and $^{55}\\rm{Mn}$",
  "authors": [
    "Nicholas A. M. Revel",
    "Baldeep Grewal",
    "Anupam Ray",
    "Aaron C. Vincent"
  ],
  "abstract": "We investigate axion-like particle (ALP) production in the interior of Betelgeuse via 

## Consistency check: LLM summary vs. raw abstracts (Betelgeuse)

Legend: ✅ supported by the abstract · ⚠️ inferred / extrapolated (not stated in the abstract) · ❌ contradicted by the abstract · ℹ️ outside knowledge flagged as such by the LLM

| # | Claim in the LLM summary | What the abstract (or tool output) says | Verdict |
|---|---|---|---|
| 1 | Spectral type M1–M2 Ia–Iab; parallax 6.55 mas | `query_simbad`: `"M1-M2Ia-Iab"`, `6.55` | ✅ |
| 2 | Axion paper (2610.34540) searches INTEGRAL and NuSTAR data for photons from ALPs produced in Betelgeuse's interior, finds no excess and sets limits on the couplings | "Using INTEGRAL-ISGRI and NuSTAR observations, we search for these photon signals, find no significant excess, and derive 95% CL exclusion limits" | ✅ |
| 3 | The ²³Na limits depend on the star's precise evolutionary stage, "which is uncertain"; the ⁵⁵Mn limits are more robust | "more sensitive to the precise evolutionary stage of Betelgeuse, whereas the ⁵⁵Mn-based limits are considerably more robust against such uncertainties" | ✅ |
| 4 | Red-nova paper (2610.31755) models a possible companion spiraling in and merging with the star | "may host a stellar companion whose orbit will decay … leading to engulfment and merger" | ✅ |
| 5 | "If the companion is confirmed, tidal decay would lead to engulfment and a merger within several thousand years" | The abstract states the decay and merger as what "recent evidence suggests"; its "If confirmed" refers to the system being an LRN progenitor | ✅ (minor rephrasing, same meaning) |
| 6 | Companions of 0.7, 1.0 and 2.0 M☉ | "companions of 0.7, 1.0, and 2.0 M☉" | ✅ |
| 7 | "In their models the merger produces a luminous red nova" | The LRN is the premise; the models follow the shock-breakout and cooling emission *preceding* the LRN, up to companion disruption | ✅ (slight simplification) |
| 8 | Ejects about 6–11 M☉ of hydrogen-rich envelope, substantially changing the envelope and circumstellar environment before core collapse | "ejects 6.15–11.19 M☉ of hydrogen-rich material, substantially modifying both the envelope structure and circumstellar environment prior to core collapse" | ✅ (rounded) |
| 9 | Interferometer paper (2609.27055) mentions Betelgeuse only as the 1921 Michelson–Pease measurement of a disk ~47 mas across, and says nothing about its evolution | "tabletop analogue of the landmark 1921 Michelson-Pease measurement of Betelgeuse … a disk 47 mas … across" | ✅ |
| 10 | "Neither is an observational study of the star. Both are theoretical." | The axion paper uses **INTEGRAL-ISGRI and NuSTAR observations**, and the summary itself says so in item 2 | ❌ |
| 11 | "Betelgeuse is an evolved red supergiant" | Not stated in any abstract; it follows from the SIMBAD output (`s*r`, luminosity class Ia–Iab) | ⚠️ (correct, but its source is the tool data, not the papers) |
| 12 | "Together, the papers treat Betelgeuse as a late-stage massive star that will eventually end in core collapse" | Only the red-nova abstract mentions core collapse ("prior to core collapse"); the axion abstract only says the evolutionary stage is uncertain | ⚠️ (generalizes one paper to "together") |

**Discrepancies flagged:**
- **#10 (❌):** the summary calls both relevant papers "theoretical", but the axion paper is an observational search (INTEGRAL-ISGRI and NuSTAR data). It is not an observation of Betelgeuse's evolution, but the label "theoretical" is wrong and contradicts the summary's own description of the paper.
- **#11 (⚠️):** "evolved red supergiant" is correct, but it comes from SIMBAD, not from the papers. The summary blends both sources without saying so.
- **#12 (⚠️):** the conclusion attributed to "the papers together" is supported by only one of them.

**Omissions (not errors):** the summary leaves out the red-nova paper's quantitative predictions (peak luminosity 2.44–6.17 × 10⁴¹ erg s⁻¹, duration 1.48–1.99 days, brighter than Venus), which are its main results.

**Overall:** the summary is faithful to the abstracts in every number and paper-specific claim (no invented values). The problems are in the synthesis: one factual error about the nature of a paper (#10) and two places where the LLM generalizes or mixes sources without flagging it (#11, #12).

# Second run: query on ton 618:

In [4]:
QUERY = ("What is the spectral type and parallax of ton 618, and what are the 3 most recent "
         "arXiv papers whose abstracts mention ton 618? Summarize what the papers suggest "
         "about its current evolutionary status.")
answer_ton618 = await run_agent(QUERY)

Starting MCP server...
Tools: ['query_simbad', 'search_arxiv']
Query: What is the spectral type and parallax of ton 618, and what are the 3 most recent arXiv papers whose abstracts mention ton 618? Summarize what the papers suggest about its current evolutionary status.

--- Turn 1 | 1.1s | in=855 out=129 | accumulated ≈ $0.0030 ---

[Tool call] query_simbad
[Input] {"object_name": "ton 618"}
[is_error] False
[Result]
{
  "object_type": "Bla",
  "spectral_type": "",
  "parallax_mas": NaN,
  "radial_velocity_km_s": 246448.6042633452
}

[Tool call] search_arxiv
[Input] {"keyword": "ton 618", "max_results": 3}
[is_error] False
[Result]
{
  "arxiv_id": "2007.03222v2",
  "title": "Stringent constraints on the light boson model with supermassive black hole spin measurements",
  "authors": [
    "Lei Zu",
    "Lei Feng",
    "Qiang Yuan",
    "Yi-Zhong Fan"
  ],
  "abstract": "Massive bosons, such as light scalars and vector bosons, can lead to instabilities of rotating black holes by the sup

## Manual verification against SIMBAD web (simbad.harvard.edu)

- **1. For Betelgeuse:**

| Field | `query_simbad` | SIMBAD web | Match? |
|---|---|---|---|
| Object type | `s*r` | Red Supergiant| Y |
| Spectral type | `M1-M2Ia-Iab` | M1-M2Ia-Iab| Y |
| Parallax (mas) | `6.55` | 6.55 [0.83] A| Y |
| Radial velocity (km/s) | `21.91` | 21.91 | Y |




- **2. For Ton 618:**



| Field | `query_simbad` | SIMBAD web | Match? |
|---|---|---|---|
| Object type | `Bla` | Blazar | Y |
| Spectral type | `""` (empty) | Not mentioned | Y |
| Parallax (mas) | `NaN` | Not mentioned | Y |
| Radial velocity (km/s) | 246448.6 | 246449 | Y |
| Redshift z | — (not returned) | z(emission) 2.2 | N |


## Consistency check: LLM summary vs. raw abstract (TON 618)

Same legend as for Betelgeuse.

| # | Claim in the LLM summary | What the abstract (or tool output) says | Verdict |
|---|---|---|---|
| 1 | Object type Bla (blazar/quasar-type AGN), "not a star" | `query_simbad`: `"Bla"`; SIMBAD web: Blazar | ✅ |
| 2 | No spectral type and no parallax (NaN), "expected for an object this distant" | `query_simbad`: `""` and `NaN` | ✅ (correct interpretation of missing data) |
| 3 | ~246,449 km/s is "a cosmological redshift, z ≈ 2.2, not a real motion through space" | The tool returned only the velocity, not z. SIMBAD web gives z = 2.2, and the relativistic Doppler formula with z = 2.2 gives exactly 246,449 km/s | ✅ (z was inferred by the LLM, and correctly) |
| 4 | The search returned only one paper, not three | `search_arxiv` returned 1 result | ✅ |
| 5 | The paper lists Ton 618 as one of the "extremely massive" supermassive black holes | "(including also the extremely massive ones OJ 287, Ton 618 and SDSS J140821.67+025733.2)" | ✅ |
| 6 | The paper uses the black hole's mass and spin to constrain light bosons (superradiance) | "we used the masses and spins of supermassive black holes … to constrain the model parameters of the light bosons", with Ton 618 in that set | ✅ |
| 7 | "If light bosons existed, they would have spun down rapidly rotating black holes like this one" | The abstract describes the spin-down mechanism for rotating black holes in general, not for Ton 618 specifically | ⚠️ (reasonable application of the general mechanism) |
| 8 | "Ton 618 is treated here as a very massive, spinning black hole" | "Extremely massive" ✅; the abstract gives no spin value for Ton 618 specifically | ⚠️ ("spinning" is implied by its use in the sample, not stated) |
| 9 | The paper is "a tool for testing particle physics, not a study of how the object is evolving" | Consistent with the abstract, which is about light bosons and fuzzy dark matter | ✅ |
| 10 | Black hole of roughly 10¹⁰ M☉, "probably in a late phase of rapid growth" | Not in the abstract; the LLM explicitly labels it as background knowledge ("treat it as background knowledge") | ℹ️ |

**Discrepancies flagged:**
- No contradictions (❌).
- **#7 and #8 (⚠️):** small extrapolations from the general mechanism to Ton 618 specifically; reasonable, but not stated in the abstract.
- **#10 (ℹ️):** the mass and growth phase come from the model's own knowledge, and it says so. "Late phase of rapid growth" is speculative. In an earlier run of the same query the model gave ~4 × 10¹⁰ M☉ instead of ~10¹⁰ M☉, so these unsourced numbers are not even stable between runs.

**Process transparency:** the LLM wrote: "I haven't tried other search terms such as "TON 618" or "Ton618"." It did not retry with alternative spellings, but it reported the limitation instead of hiding it.

**Overall:** the summary is consistent with the single abstract it received. It correctly separates what comes from the paper from outside knowledge, and it correctly interprets the ill-defined SIMBAD output (failure mode 2) instead of reporting `NaN` or 0.82 c as real stellar values.

## Physical plausibility check

### Betelgeuse

| Quantity | Tool value | Check | Plausible? |
|---|---|---|---|
| Parallax → distance | 6.55 ± 0.83 mas (SIMBAD web gives the error; the tool does not) | $d = 1/\varpi = 1000/6.55 \approx 153$ pc, range 136–175 pc for $\varpi \pm \sigma$ (~13%) | ✅ A few hundred light-years (~500 ly), as expected for one of the brightest stars in the sky |
| Spectral type vs. object type | `M1-M2Ia-Iab` / `s*r` | Spectral class M = cool (~3,500 K); luminosity class Ia–Iab = supergiant → red supergiant, which is exactly what `s*r` means | ✅ Internally consistent |
| Radial velocity | 21.91 km/s | Typical values for stars in the Galactic disk are tens of km/s | ✅ |
| Cross-check with the arXiv data | 47 mas angular diameter (Michelson–Pease, in the interferometer abstract) | $R = \frac{\theta}{2}\,d = 0.0235'' \times 153\ \text{pc} \approx 3.6\ \text{AU} \approx 770\ R_\odot$ | ✅ Hundreds of solar radii, as expected for a red supergiant |

### TON 618

| Quantity | Tool value | Check | Plausible? |
|---|---|---|---|
| Velocity vs. redshift | 246,448.6 km/s (tool) and z = 2.2 (SIMBAD web) | Relativistic Doppler: $v = c\,\frac{(1+z)^2-1}{(1+z)^2+1} = 0.822\,c \approx 246{,}449$ km/s, exactly the tool value. The non-relativistic $cz = 2.2\,c \approx 659{,}543$ km/s would exceed the speed of light | ✅ $v < c$, but it is a **cosmological recession velocity**, not a stellar radial velocity. At z = 2.2 the Doppler formula is not even the right physical description (the redshift comes from the expansion of the universe), so reporting z is more meaningful |
| Parallax | `NaN` (no value in SIMBAD) | At z = 2.2 the comoving distance is ~5.6 Gpc (Planck 2018 cosmology), so the parallax would be of order $10^{-7}$ mas, many orders of magnitude below Gaia's precision (~0.01–0.1 mas) | ✅ The absence of a parallax is physically expected |
| Spectral type | `""` (empty) | MK spectral types (O, B, A, F, G, K, M) classify stars; a quasar's spectrum is dominated by emission from the accretion disk around the black hole | ✅ The absence of a spectral type is physically expected |

**Conclusion:** all values returned by `query_simbad` for Betelgeuse are physically plausible and mutually consistent. For TON 618, the values are correct as SIMBAD data, but their labels are misleading for an extragalactic object (a "radial velocity" that is really a redshift, and `NaN`/`""` with no explanation). The plausibility check catches this; the tool does not (see failure mode 2 in `week05_failures_Aguilar_Mayorga.md`).

# Original code version, reported as error #3 in week05_failures_Aguilar_Mayorga.md:

```
async def run_agent(query: str) -> str:                     # ← LLM receives the query
    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    params = StdioServerParameters(command=sys.executable, args=[str(SERVER_SCRIPT)])

    #async with stdio_client(params) as (read, write):
    print("Starting MCP server...", flush = True)
    async with stdio_client(params, errlog = open("server_stderr.log","w")) as (read, write): #changed this line to keep track of whether the connection has been well-stablished
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = (await session.list_tools()).tools
            schemas = [{"name": t.name, "description": t.description or "",
                        "input_schema": t.input_schema} for t in tools]
            print(f"Tools: {[t.name for t in tools]}\nQuery: {query}\n{'='*60}")

            messages = [{"role": "user", "content": query}]
            while True:
                response = client.messages.create(
                    model=MODEL, max_tokens=2048,                # ← increased tokens limit so abstract would fit complete
                    tools=schemas, messages=messages)
                messages.append({"role": "assistant", "content": response.content})

                if response.stop_reason != "tool_use":
                    final = "".join(b.text for b in response.content if b.type == "text")
                    print(f"\n[stop_reason: {response.stop_reason}]\n\nFINAL ANSWER:\n{final}")
                    return final

                tool_results = []
                for block in response.content:
                    if block.type != "tool_use":
                        continue
                    print(f"\n[Tool call] {block.name}\n[Input] {json.dumps(block.input)}")
                    result = await session.call_tool(block.name, block.input)
                    text = "\n".join(c.text for c in result.content if hasattr(c, "text"))
                    print(f"[is_error] {result.is_error}\n[Result]\n{text}") 
                    tool_results.append({"type": "tool_result", "tool_use_id": block.id,
                                         "content": text, "is_error": result.is_error})
                messages.append({"role": "user", "content": tool_results})

```
